# core

> `astream` and `ClaudeRun`: stateless completions through the installed Claude Code

In [ ]:
#| default_exp core

`astream(msgs, ...)` returns a `ClaudeRun`: one stateless completion through the installed `claude`, using its login and subscription. The complete history (as `aidialog.msg_parts.Msg`s, ending with a real user prompt) is compiled into a native transcript via `fastclaude.session` under a fresh random session id and resumed; callable tools are served in-process via `fastclaude.protocol`; iteration yields the raw stream-json events; `.messages` accumulates the canonical trace (signatures intact, tool names unqualified) and `.result` the terminal result, so the next request can replay everything. `run.interrupt()` ends a turn natively, and closing the stream interrupts, drains, escalates to terminate/kill, and removes the transcript. Runs default to an isolated XDG cache work dir, so no real project's sessions or settings are touched; pass `cwd=` for project context, and `native_tools=` to enable built-ins such as `'WebSearch'`.

In [ ]:
#| export
import asyncio, json, os, shutil, uuid
from contextlib import suppress
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.xdg import xdg_cache_home
from fastllm.anthropic import denorm_msgs, norm_parts
from fastllm.types import unwrap_typed
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult, Media
from fastclaude.session import *
from fastclaude.protocol import *

In [ ]:
from fastcore.test import *
import stat, sys, tempfile, textwrap

## The work dir

In Claude Code, the working directory is a storage key: the transcript we fabricate must sit in the `~/.claude/projects` folder derived from the directory the process runs in, or `--resume` finds nothing. It is also a behavior input, deciding which project settings and CLAUDE.md join the run, and whose session list our synthetic transcripts would pollute. So by default each run gets an isolated pseudo-project under the XDG cache dir: answers stop depending on where the host process happens to sit, no real project's sessions are touched, and everything filed under the cache path's project folder is ours to delete. Passing `cwd=` opts in to a real project's context instead.

In [ ]:
#| export
MCP_SERVER = 'fastclaude'
MCP_PREFIX = f'mcp__{MCP_SERVER}__'
SERVER_TOOLS = ('WebSearch','WebFetch')

def work_dir():
    "The default run directory: an isolated pseudo-project under the XDG cache dir"
    p = xdg_cache_home()/'fastclaude'
    p.mkdir(parents=True, exist_ok=True)
    return p

## Compiling history

Every call supplies the complete history as `aidialog.msg_parts.Msg` objects, so a caller can edit, hide, rewind, or replace anything before replaying it. The sequence must end with a genuine user prompt: text (or media), not a bare `ToolResult`, since Claude owns the live tool loop and a mid-loop continuation is not a completion request. `denorm_msgs` converts the history to Anthropic-style wire messages, and `prefix_tools` qualifies past callable-tool names the way this run will offer them, leaving Claude Code's own tool names alone. The final message's content becomes the live turn; everything before it becomes the transcript.

In [ ]:
#| export
def compile_msgs(
    msgs, # Complete history as `Msg`s, ending with a real user prompt
):
    "`(history, prompt)`: qualified wire messages for the transcript, and the live turn's content"
    msgs = listify(msgs)
    if not msgs: raise ValueError('empty message history')
    last = msgs[-1]
    if last.role!='user' or not any(isinstance(p, (Text,Media)) for p in last.content):
        raise ValueError('history must end with a user message containing a real prompt, not only tool results')
    den = denorm_msgs(msgs)
    return prefix_tools(den[:-1], MCP_PREFIX, skip=SERVER_TOOLS), den[-1]['content']

In [ ]:
# chkstyle: ignore-node
h = [Msg('user', [Text('Measure the flux please.')]),
    Msg('assistant', [Text('Checking.'), ToolUse(id='t1', name='flux_meter', arguments={})]),
    Msg('tool', [ToolResult(id='t1', name='flux_meter', text='flux: 41.7 kf')]),
    Msg('user', [Text('And in gauss?')])]
hist,prompt = compile_msgs(h)
test_eq(len(hist), 3)
test_eq(hist[1]['content'][1]['name'], 'mcp__fastclaude__flux_meter')
test_eq(prompt, [dict(type='text', text='And in gauss?')])
hist[1]

A history ending only in a `ToolResult` is the external-tool-loop shape, and it is rejected clearly rather than half-supported; regeneration from that boundary is a possible later option (see DEV.md), not a completion:

In [ ]:
with expect_fail(ValueError, contains='real prompt'): compile_msgs(h[:3])
with expect_fail(ValueError, contains='real prompt'): compile_msgs(h[:2])
with expect_fail(ValueError, contains='empty'): compile_msgs([])

## The command and environment

The executable is the user's installed `claude`, with their real config and login: that is the whole point, and it is why `CLAUDE_CONFIG_DIR` is never isolated. `--strict-mcp-config` is always passed, keeping their other configured MCP servers out of every run (without it, a run with no tools of its own will happily use whatever MCP servers the user has configured), `--tools` with an explicit list (empty by default) disables built-in tools until asked for, and the private SDK server entry in `--mcp-config` is how our in-process tools join. `--resume` uses the equals form so a dash-leading session id can never parse as a flag of its own. The environment blanks `ANTHROPIC_API_KEY`, so an inherited key cannot silently bill the API, and removes `CLAUDECODE`, so the child does not believe it is nested inside another Claude Code.

In [ ]:
#| export
def claude_cmd(
    model=None, # Model alias or full name; None uses the user's default
    resume=None, # Session id to resume, i.e. the transcript just written
    system=None, # System prompt; None keeps Claude Code's own
    tools=False, # Offer the private SDK MCP server?
    native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
    allowed=(), # `--allowedTools` entries, e.g. qualified callable names
    claude_path=None, # Explicit claude executable; found on PATH if None
):
    "argv for one headless stream-json claude run"
    c = [str(claude_path or shutil.which('claude') or 'claude'), '--output-format','stream-json',
        '--input-format','stream-json', '--verbose', '--include-partial-messages']
    if model: c += ['--model', model]
    if system is not None: c += ['--system-prompt', system]
    if resume: c += [f'--resume={resume}']
    if tools: c += ['--mcp-config', json.dumps(dict(mcpServers={MCP_SERVER: dict(type='sdk', name=MCP_SERVER)}))]
    c.append('--strict-mcp-config')
    c += ['--tools', ','.join(native_tools)]
    if allowed: c += ['--allowedTools', ','.join(allowed)]
    return c

def claude_env():
    "A child environment that cannot bill an API key and does not think it is nested"
    env = dict(os.environ, ANTHROPIC_API_KEY='')
    env.pop('CLAUDECODE', None)
    return env

In [ ]:
c = claude_cmd('sonnet', resume='-abc', tools=True, allowed=['mcp__fastclaude__flux_meter'])
test('--resume=-abc', c, in_)
test_eq(c[c.index('--tools')+1], '')
test('--strict-mcp-config', c, in_)
test('--strict-mcp-config', claude_cmd('sonnet'), in_)
test_eq(json.loads(c[c.index('--mcp-config')+1]), dict(mcpServers=dict(fastclaude=dict(type='sdk', name='fastclaude'))))
env = claude_env()
test_eq(env['ANTHROPIC_API_KEY'], '')
assert 'CLAUDECODE' not in env
c[1:]

## The run

A `ClaudeRun` is request-scoped: it owns exactly one temporary transcript and one process, then becomes terminal. Iterating it drives the whole lifecycle: compile and file the history under a fresh random session id, spawn `claude` resuming it, shake hands, send the live turn, and yield every non-control event raw. Along the way it accumulates the canonical trace: each full `assistant` event becomes a `Msg` via `norm_parts` (signatures intact, callable names unqualified), each tool-result `user` event a `tool` role `Msg`, and the terminal `result` lands on `.result`. A later stateless request can therefore receive this run's tool uses and results, not merely its final text.

In [ ]:
#| export
class ClaudeRun:
    "One stateless completion: a fresh transcript, one claude process, streamed events, a canonical trace"
    def __init__(self,
        msgs, # Complete history as `Msg`s, ending with a real user prompt
        model='sonnet', # Model alias or full name
        system=None, # System prompt; None keeps Claude Code's own
        tools=None, # Callable tools, in either `tool_spec` form
        cwd=None, # Project directory for the run; the isolated `work_dir()` if None
        native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
        claude_path=None, # Explicit claude executable; found on PATH if None
    ):
        store_attr()
        self.messages,self.result,self.proc,self.proto = [],None,None,None
        self._names,self._closed,self._spath = {},False,None

    def __aiter__(self):
        if not hasattr(self, '_it'): self._it = self._run()
        return self._it

@delegates(ClaudeRun)
def astream(msgs, **kwargs):
    "Start one stateless completion; iterate the returned `ClaudeRun` for its raw events"
    return ClaudeRun(msgs, **kwargs)

Spawning files the history and builds the process. A fresh random session id avoids collisions when identical histories run concurrently; prompt caching depends on request content, not on session reuse. An empty history (a bare first prompt) writes no transcript and resumes nothing:

In [ ]:
#| export
@patch
async def _spawn(self:ClaudeRun):
    "Compile and file the history, then start claude resuming it; returns the live turn's content"
    self.cwd = Path(self.cwd) if self.cwd else work_dir()
    hist,prompt = compile_msgs(self.msgs)
    sid = str(uuid.uuid4())
    if hist:
        save_sess(msgs2recs(hist, key=sid, cwd=self.cwd), sid, self.cwd)
        self._spath = sess_file(sid, self.cwd)
    schemas,_ = mk_tools(self.tools or [])
    allowed = [MCP_PREFIX+s['name'] for s in schemas] + list(self.native_tools)
    argv = claude_cmd(self.model, resume=sid if hist else None, system=self.system, tools=bool(schemas),
        native_tools=self.native_tools, allowed=allowed, claude_path=self.claude_path)
    self.proc = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=self.cwd, env=claude_env())
    self.proto = ClaudeProto(self.proc, tools=self.tools, server=MCP_SERVER)
    return prompt

Trace accumulation reads only the full message events, ignoring partials: they are transcript-grade, one event per content block. Callable names are unqualified on the way back, so the caller sees the tool it registered, not Claude's `mcp__` spelling. Result text passes through `unwrap_typed`, restoring the `str` subclass that `tool_content` marked, so a `FullResponse` reaches `.messages` with its no-truncation meaning intact. `user` events that are not tool results (such as the native interrupt record) still stream through raw, but are not part of the generated trace:

In [ ]:
#| export
def _unq(nm): return nm[len(MCP_PREFIX):] if nm and nm.startswith(MCP_PREFIX) else nm
def _flat(c): return c if isinstance(c, str) else '\n'.join(b.get('text','') for b in c if b.get('type')=='text')

@patch
def _track(self:ClaudeRun, m):
    "Fold one raw event into `.messages` and `.result`"
    t,c = m.get('type'), nested_idx(m, 'message', 'content')
    if t=='assistant' and isinstance(c, list):
        parts = norm_parts(m['message'])
        for p in parts:
            if isinstance(p, ToolUse): p.name = self._names[p.id] = _unq(p.name)
        self.messages.append(Msg('assistant', parts))
    elif t=='user' and isinstance(c, list) and c and all(b.get('type')=='tool_result' for b in c):
        self.messages.append(Msg('tool', [ToolResult(id=b.get('tool_use_id'), name=self._names.get(b.get('tool_use_id')),
            text=unwrap_typed(_flat(b.get('content','')))) for b in c]))
    elif t=='result': self.result = m

The drive: a kickoff task completes the handshake and sends the live turn (its response arrives through the same read loop the consumer iterates), events stream through `_track` to the caller, and the terminal result ends the run. However the loop ends - result, consumer break, cancellation - `aclose` runs:

In [ ]:
#| export
@patch
async def _kick(self:ClaudeRun, prompt):
    "Handshake, then the one live user turn"
    await self.proto.initialize()
    await self.proto.send(dict(type='user', message=dict(role='user', content=prompt)))

@patch
async def _run(self:ClaudeRun):
    "The event stream: spawn, kick off, yield raw events until the terminal result"
    prompt = await self._spawn()
    kick = asyncio.create_task(self._kick(prompt))
    try:
        async for m in self.proto.events():
            self._track(m)
            yield m
            if m.get('type')=='result': break
    finally:
        kick.cancel()
        await self.aclose()

@patch
async def interrupt(self:ClaudeRun, timeout=30):
    "Claude's native interrupt: end the current turn; the stream stays open to drain the aborted tail"
    return await self.proto.interrupt(timeout)

Cleanup is the part that must work under cancellation, because a host's ctrl-C arrives as exactly that: the consumer task is cancelled, the stream is closed, and this sequence is all that stands between an abandoned turn and an orphaned process. If the turn is still running, send the native interrupt and drain the tail directly (the consumer's read loop is gone, so `aclose` reads for itself, still folding what arrives into the trace). Then close stdin, wait, and escalate: terminate, then kill, each with a deadline. Finally remove the exact transcript this run wrote, success or failure. The whole body runs shielded, so a second cancellation cannot abort it partway:

In [ ]:
#| export
@patch
async def _drain(self:ClaudeRun, timeout=10):
    "Read the aborted turn's tail directly, resolving control responses and folding events into the trace"
    async def _go():
        async for m in read_msgs(self.proc.stdout):
            if m.get('type')=='control_response': self.proto._resolve(m)
            else: self._track(m)
            if m.get('type')=='result': return
    with suppress(Exception): await asyncio.wait_for(_go(), timeout)

@patch
async def _cleanup(self:ClaudeRun):
    p = self.proc
    if p and p.returncode is None:
        if self.result is None and p.stdin and not p.stdin.is_closing():
            t = asyncio.create_task(self._drain())
            with suppress(Exception): await asyncio.wait_for(self.proto.interrupt(), 5)
            await t
        with suppress(Exception): p.stdin.close()
        try: await asyncio.wait_for(p.wait(), 5)
        except (TimeoutError, asyncio.TimeoutError):
            with suppress(ProcessLookupError): p.terminate()
            try: await asyncio.wait_for(p.wait(), 5)
            except (TimeoutError, asyncio.TimeoutError):
                with suppress(ProcessLookupError): p.kill()
                with suppress(Exception): await p.wait()
    if self.proto: await self.proto.aclose()
    if self._spath: Path(self._spath).unlink(missing_ok=True)

@patch
async def aclose(self:ClaudeRun):
    "Interrupt if mid-turn, drain, close, escalate, and remove the transcript; idempotent and cancellation-shielded"
    if self._closed: return
    self._closed = True
    await asyncio.shield(asyncio.create_task(self._cleanup()))

## A scripted run

The lifecycle runs end to end against a scripted stand-in for `claude`, so the run's own logic - kickoff ordering, trace accumulation, terminal handling, transcript cleanup - is verified offline. The script answers any control request, echoes the user turn as an assistant message, and finishes with a result:

In [ ]:
# chkstyle: skip
fake_src = textwrap.dedent('''
    #!/usr/bin/env python3
    import sys, json
    def w(o): sys.stdout.write(json.dumps(o)+'\\n'); sys.stdout.flush()
    for line in sys.stdin:
        m = json.loads(line)
        if m.get('type')=='control_request':
            w(dict(type='control_response', response=dict(subtype='success', request_id=m['request_id'], response={})))
        elif m.get('type')=='user':
            c = m['message']['content']
            txt = 'echo: '+(c if isinstance(c, str) else c[0].get('text',''))
            w(dict(type='assistant', message=dict(role='assistant', content=[dict(type='text', text=txt)])))
            w(dict(type='result', subtype='success', result=txt))
    ''').strip()
fake_cc = Path(tempfile.mkdtemp())/'claude'
fake_cc.write_text(fake_src+'\n')
fake_cc.chmod(fake_cc.stat().st_mode | stat.S_IXUSR)

In [ ]:
scratch = Path(tempfile.mkdtemp())
run = astream(h, claude_path=fake_cc, cwd=scratch)
got = [m async for m in run]
test_eq(run.result['result'], 'echo: And in gauss?')
test_eq([m.role for m in run.messages], ['assistant'])
test_eq(run.messages[0].content[0].text, 'echo: And in gauss?')
test_eq(run._spath.exists(), False)
[m['type'] for m in got]

## Live runs

The proofs below ran against the real authenticated CLI (each output shown is a genuine capture; they spend tokens, so they stay out of automated runs). First the whole point of the design in one cell: Claude calls an in-process Python callable through the bridge, continues its own loop, and the trace comes back canonical - per-block `Msg`s, names unqualified, the thinking block's signature intact in `raw`:

In [ ]:
#| eval: false
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

lmsgs = [Msg('user', [Text('Use the flux_meter tool with unit="gauss", then reply with exactly the tool output.')])]
lrun = astream(lmsgs, tools=[flux_meter])
levs = [m async for m in lrun]
test_eq(lrun.result['result'], 'flux: 41.7 gauss')
test_eq(lrun.messages[-2].content[0].name, 'flux_meter')
test_eq(lrun.messages[-2].content[0].text, 'flux: 41.7 gauss')
[(m.role, [type(p).__name__ for p in m.content]) for m in lrun.messages]

[('assistant', ['Thinking']),
 ('assistant', ['ToolUse']),
 ('tool', ['ToolResult']),
 ('assistant', ['Text'])]

Interruption is the load-bearing path: with Claude owning the tool loop, the host's ctrl-C must stop a run mid-tool. `run.interrupt()` sends the native control request; a host closing the stream gets the same sequence via `aclose`. Interrupted mid-call, Claude sends `control_cancel_request` for the pending tool, so the callable sees an ordinary `CancelledError` (a kernel-backed tool translates that into a kernel interrupt), Claude files a result for the aborted call, and the terminal result reports `aborted_tools`. The caller's explicit interrupt is thus visible in `.result`, distinct from a provider failure:

In [ ]:
#| eval: false
seen = []
async def slow_probe() -> str:
    "Measure very slowly."
    try:
        await asyncio.sleep(60)
        return 'done'
    except asyncio.CancelledError:
        seen.append('cancelled')
        raise

In [ ]:
#| eval: false
irun = astream([Msg('user', [Text('Use the slow_probe tool, then report its output.')])], tools=[slow_probe])
async for m in irun:
    if m.get('type')=='assistant' and any(b.get('type')=='tool_use' for b in (m['message'].get('content') or [])):
        asyncio.get_running_loop().call_later(1, lambda: asyncio.ensure_future(irun.interrupt()))
test_eq(seen, ['cancelled'])
test_eq(irun.result['is_error'], True)
{k: irun.result.get(k) for k in ('subtype','terminal_reason')}

{'subtype': 'error_during_execution', 'terminal_reason': 'aborted_tools'}

Statelessness closes the loop: the next request passes the previous run's trace as ordinary history. The signed thinking block replays into the transcript, the tool exchange is there to be referred to, and nothing re-executes:

In [ ]:
#| eval: false
cmsgs = lmsgs + lrun.messages + [Msg('user', [Text('What unit did the flux_meter report in? One word only.')])]
crun = astream(cmsgs, tools=[flux_meter])
cevs = [m async for m in crun]
crun.result['result']

'Gauss'

## Cleanup

Runs remove their own transcripts, so only empty per-run project folders remain; the scripted run's scratch folder is ours to delete, and the shared cache-dir folder is scratch by contract.

In [ ]:
shutil.rmtree(sess_dir(scratch), ignore_errors=True)
shutil.rmtree(sess_dir(work_dir()), ignore_errors=True)
shutil.rmtree(fake_cc.parent, ignore_errors=True)
shutil.rmtree(scratch, ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()